In [1]:
import os
from datetime import date, datetime
from sqlalchemy import create_engine, text

DATABASE_URL = os.environ["DATABASE_URL"]
OLLAMA_BASE_URL = os.environ["OLLAMA_BASE_URL"]
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "nomic-embed-text")

engine = create_engine(DATABASE_URL, future=True)
print("Database URL loaded.")
print("Embedding model:", EMBEDDING_MODEL)

Database URL loaded.
Embedding model: nomic-embed-text


In [2]:
ddl = """
CREATE EXTENSION IF NOT EXISTS vector;

CREATE TABLE IF NOT EXISTS properties (
    id TEXT PRIMARY KEY,
    name TEXT NOT NULL,
    address TEXT NOT NULL,
    city TEXT NOT NULL,
    amenities TEXT NOT NULL,
    monthly_rent NUMERIC(12,2) NOT NULL,
    created_at TIMESTAMP NOT NULL DEFAULT NOW()
);

CREATE TABLE IF NOT EXISTS tenants (
    id TEXT PRIMARY KEY,
    full_name TEXT NOT NULL,
    email TEXT,
    phone TEXT,
    created_at TIMESTAMP NOT NULL DEFAULT NOW()
);

CREATE TABLE IF NOT EXISTS leases (
    id TEXT PRIMARY KEY,
    property_id TEXT NOT NULL REFERENCES properties(id),
    tenant_id TEXT NOT NULL REFERENCES tenants(id),
    start_date DATE NOT NULL,
    end_date DATE NOT NULL,
    monthly_rent NUMERIC(12,2) NOT NULL,
    status TEXT NOT NULL,
    created_at TIMESTAMP NOT NULL DEFAULT NOW()
);

CREATE TABLE IF NOT EXISTS payments (
    id TEXT PRIMARY KEY,
    lease_id TEXT NOT NULL REFERENCES leases(id),
    due_date DATE NOT NULL,
    paid_date DATE,
    amount NUMERIC(12,2) NOT NULL,
    status TEXT NOT NULL,
    created_at TIMESTAMP NOT NULL DEFAULT NOW()
);

CREATE TABLE IF NOT EXISTS maintenance_logs (
    id TEXT PRIMARY KEY,
    property_id TEXT NOT NULL REFERENCES properties(id),
    tenant_id TEXT REFERENCES tenants(id),
    issue TEXT NOT NULL,
    priority TEXT NOT NULL,
    status TEXT NOT NULL,
    reported_at TIMESTAMP NOT NULL DEFAULT NOW(),
    resolved_at TIMESTAMP
);

CREATE TABLE IF NOT EXISTS conversation_events (
    id TEXT PRIMARY KEY,
    tenant_id TEXT REFERENCES tenants(id),
    property_id TEXT REFERENCES properties(id),
    agent_type TEXT NOT NULL,
    user_message TEXT NOT NULL,
    assistant_message TEXT NOT NULL,
    created_at TIMESTAMP NOT NULL DEFAULT NOW()
);
"""

with engine.begin() as conn:
    conn.execute(text(ddl))

print("Schema created.")

Schema created.


In [3]:

seed_sql = """
INSERT INTO properties (id, name, address, city, amenities, monthly_rent)
VALUES
('prop_001', 'Sunrise Residency - Room 2', '02 Roeland Street', 'Cape Town', 'WiFi, Parking, Ensuite Bathroom, 24/7 Water', 8500.00)
ON CONFLICT (id) DO NOTHING;

INSERT INTO tenants (id, full_name, email, phone)
VALUES
('ten_001', 'Bennet Dyani', 'bennet@gmail.com', '0630732937')
ON CONFLICT (id) DO NOTHING;

INSERT INTO leases (id, property_id, tenant_id, start_date, end_date, monthly_rent, status)
VALUES
('lease_001', 'prop_001', 'ten_001', '2026-01-01', '2026-12-31', 8500.00, 'active')
ON CONFLICT (id) DO NOTHING;

INSERT INTO payments (id, lease_id, due_date, paid_date, amount, status)
VALUES
('pay_001', 'lease_001', '2026-06-05', '2026-06-03', 8500.00, 'paid'),
('pay_002', 'lease_001', '2026-07-05', NULL, 8500.00, 'pending')
ON CONFLICT (id) DO NOTHING;

INSERT INTO maintenance_logs (id, property_id, tenant_id, issue, priority, status)
VALUES
('mnt_001', 'prop_001', 'ten_001', 'Leaking kitchen tap', 'medium', 'open')
ON CONFLICT (id) DO NOTHING;
"""

with engine.begin() as conn:
    conn.execute(text(seed_sql))

print("Seed data inserted.")

Seed data inserted.


In [4]:

checks = {
    "properties": "SELECT COUNT(*) FROM properties;",
    "tenants": "SELECT COUNT(*) FROM tenants;",
    "leases": "SELECT COUNT(*) FROM leases;",
    "payments": "SELECT COUNT(*) FROM payments;",
    "maintenance_logs": "SELECT COUNT(*) FROM maintenance_logs;",
}

with engine.connect() as conn:
    for name, q in checks.items():
        count = conn.execute(text(q)).scalar_one()
        print(f"{name}: {count}")

properties: 1
tenants: 1
leases: 1
payments: 2
maintenance_logs: 1


In [5]:
from langchain_core.documents import Document

docs = [
    Document(
        page_content="Room 2 at Sunrise Residency is currently available for new tenants from next month.",
        metadata = {"property_id": "prop_001", "topic": "availability", "source": "property_sheet"}
        
    ),

    Document(
        page_content= "Monthly rent Room 2 is 8500.00 R. Security deposit is one month rent.",
        
    ),
    Document(
        page_content="Amenities include high-speed WIFI, parking, ensute bathroom, and 24/7 water supply.",
        metadata= {"property_id": "prop_001", "topic": "amenities", "source": "amenities_guide"}
        
    ),
]
print("Prepared docs:", len(docs))

Prepared docs: 3


In [6]:
# Cell 6: Ingest docs into pgvector via LangChain
from langchain_ollama import OllamaEmbeddings
from langchain_postgres import PGVector

embeddings = OllamaEmbeddings(
    model=EMBEDDING_MODEL,
    base_url=OLLAMA_BASE_URL
)

vector_store = PGVector(
    embeddings=embeddings,
    collection_name="rental_knowledge",
    connection=DATABASE_URL,
    use_jsonb=True,
)

vector_store.add_documents(docs)
print("RAG docs ingested into pgvector collection: rental_knowledge")

RAG docs ingested into pgvector collection: rental_knowledge
